# Week 6: From Random Forest to LightGBM

## Learning Objectives

1. Understand Random Forest baseline (parallel independent trees from Week 5)
2. Learn Gradient Boosting concept (sequential trees that correct errors)
3. Master LightGBM (modern leaf-wise boosting with hyperparameter optimization)
4. Evaluate with 4 metrics (Accuracy, F1, ROC-AUC, MCC)
5. Get hands-on practice (hyperparameter tuning with Optuna)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 21

## Evaluation Metrics

We use four metrics following Chicco & Jurman (2020):

| Metric | Interpretation | When to Use |
|--------|-----------------|-------------|
| Accuracy | % of correct predictions | Overall performance |
| F1-Score | Balance false positives and false negatives | Imbalanced data |
| ROC-AUC | Probability model ranks positive higher than negative | Threshold-independent |
| MCC | Correlation between predicted and actual | Most robust for imbalanced data |

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, matthews_corrcoef

df = pd.read_csv('heart_failure_clinical_records_dataset.csv')
df.drop(columns=['time'], inplace=True)
X = df.drop(columns=['DEATH_EVENT'])
y = df['DEATH_EVENT']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [3]:
def evaluate_model(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_te, y_pred),
        'F1': f1_score(y_te, y_pred),
        'ROC-AUC': roc_auc_score(y_te, y_proba),
        'MCC': matthews_corrcoef(y_te, y_pred)
    }

all_results = []

## Random Forest Baseline (Week 5 Review)

Many independent trees vote on the prediction (parallel):

- Bootstrap: Each tree trained on random sample of data (with replacement)
- Random features: Each split considers only √p features
- Average votes: Majority vote for classification

Averaging uncorrelated high-variance trees reduces variance

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_baseline = RandomForestClassifier(
    n_estimators=50, max_depth=10, max_features='sqrt',
    min_samples_split=5, random_state=RANDOM_STATE
)
rf_baseline.fit(X_train_s, y_train)

rf_result = evaluate_model('Random Forest (Baseline)', rf_baseline, X_test_s, y_test)
all_results.append(rf_result)

print("Random Forest Baseline Performance")
for metric, value in rf_result.items():
    if metric != 'Model':
        print(f"  {metric:10s}: {value:.4f}")

Random Forest Baseline Performance
  Accuracy  : 0.7333
  F1        : 0.6000
  ROC-AUC   : 0.7598
  MCC       : 0.4008


## Gradient Boosting (Sequential Learning)

Trees learn from previous errors (sequential):

- Iteration 1: Train first tree on data
- Iteration 2+: Train next tree on residuals (errors) from all previous trees
- Combine: prediction = tree1 + lr × tree2 + lr × tree3 + ...

Each tree focuses on fixing remaining mistakes, which reduces both bias and variance

Key differences from Random Forest:

| Aspect | Random Forest | Gradient Boosting |
|---|---|---|
| Process | Parallel independent trees | Sequential error correction |
| Tree depth | Deep (10+) | Shallow (3-5) |
| Learning | Variance reduction | Bias + Variance reduction |
| Speed | Fast | Slower (sequential) |
| Accuracy | Good | Often better |

In [5]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    subsample=0.8, random_state=RANDOM_STATE
)
gb.fit(X_train_s, y_train)

gb_result = evaluate_model('Gradient Boosting', gb, X_test_s, y_test)
all_results.append(gb_result)

print("Gradient Boosting Performance:")
for metric, value in gb_result.items():
    if metric != 'Model':
        print(f"  {metric:10s}: {value:.4f}")

print("\nImprovement over Random Forest:")
for metric in ['Accuracy', 'F1', 'ROC-AUC', 'MCC']:
    improvement = gb_result[metric] - rf_result[metric]
    sign = "+" if improvement >= 0 else ""
    print(f"  {metric:10s}: {sign}{improvement:+.4f}")

Gradient Boosting Performance:
  Accuracy  : 0.7222
  F1        : 0.5614
  ROC-AUC   : 0.7049
  MCC       : 0.3584

Improvement over Random Forest:
  Accuracy  : -0.0111
  F1        : -0.0386
  ROC-AUC   : -0.0548
  MCC       : -0.0425


## Exercise 1: Learning Rate Impact

Task: Train Gradient Boosting with `learning_rate=0.05` and compare to 0.1

Theory: 
- High lr (0.1) = each tree contributes 10% → faster learning
- Low lr (0.05) = each tree contributes 5% → slower but more stable

What to observe: Does slower learning improve test performance?

In [6]:
gb_low_lr = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=3,
    subsample=0.8, random_state=RANDOM_STATE
)
gb_low_lr.fit(X_train_s, y_train)

gb_low_lr_result = evaluate_model('Gradient Boosting (lr = 0.05)', gb_low_lr, X_test_s, y_test)
all_results.append(gb_low_lr_result)

print("Gradient Boosting (lr = 0.05) Performance:")
for metric, value in gb_low_lr_result.items():
    if metric != 'Model':
        print(f"  {metric:10s}: {value:.4f}")

Gradient Boosting (lr = 0.05) Performance:
  Accuracy  : 0.7000
  F1        : 0.5574
  ROC-AUC   : 0.7275
  MCC       : 0.3322


## LightGBM (Fast Modern Boosting)

How it differs: Use leaf-wise growth instead of level-wise

- Traditional: Split all nodes at each level (like a full binary tree)
- LightGBM: Split only the node that reduces loss most (smart focus)

Result: 10-20× faster, fewer trees needed, same/better accuracy

Key hyperparameters:
- `num_leaves`: Max leaves per tree (higher = more complex, higher variance)
- `learning_rate`: Same as GB (0.1 = standard, 0.05 = slower/stable)
- `min_data_in_leaf`: Minimum samples required in each leaf (prevent overfitting)
- `feature_fraction`: Fraction of features per tree (0.8 = use 80%)
- `bagging_fraction`: Fraction of samples per tree (0.8 = use 80%)

Caution: Leaf-wise can overfit on small data so use higher `min_data_in_leaf` (20-30)

In [7]:
from lightgbm import LGBMClassifier

lgbm_simple = LGBMClassifier(
    n_estimators=100, learning_rate=0.1, num_leaves=31,
    verbose=-1, random_state=RANDOM_STATE
)
lgbm_simple.fit(X_train_s, y_train)

lgbm_simple_result = evaluate_model('LightGBM (Simple)', lgbm_simple, X_test_s, y_test)
all_results.append(lgbm_simple_result)

print("LightGBM (Simple) Performance:")
for metric, value in lgbm_simple_result.items():
    if metric != 'Model':
        print(f"  {metric:10s}: {value:.4f}")

LightGBM (Simple) Performance:
  Accuracy  : 0.7000
  F1        : 0.5424
  ROC-AUC   : 0.6885
  MCC       : 0.3194


## Hyperparameter Tuning with Optuna

Why Optuna: Bayesian optimization finds good hyperparameters automatically

Why log-scale for learning_rate: Exponential effects (0.001, 0.01, 0.1, 0.3 are equally spaced)

Why ROC-AUC: Handle imbalanced data better than accuracy

In [8]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 30)
    }
    model = LGBMClassifier(**params, verbose=-1, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='roc_auc')
    return scores.mean()

sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f"\nBest ROC-AUC (5-fold CV): {study.best_value:.4f}")
print(f"Best Hyperparameters:")
for key, val in sorted(study.best_params.items()):
    print(f"  {key:20s}: {val}")

[I 2026-08-15 12:58:17,554] A new study created in memory with name: no-name-1dd9c4aa-890b-4f7c-9aac-cf5b0e0c3300


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-15 12:58:17,742] Trial 0 finished with value: 0.7784597791371188 and parameters: {'n_estimators': 54, 'learning_rate': 0.019458513490035873, 'num_leaves': 40, 'max_depth': 3, 'feature_fraction': 0.7617768295802316, 'bagging_fraction': 0.715231977008613, 'min_data_in_leaf': 16}. Best is trial 0 with value: 0.7784597791371188.
[I 2026-08-15 12:58:18,109] Trial 1 finished with value: 0.7891788014940725 and parameters: {'n_estimators': 117, 'learning_rate': 0.02032892404610392, 'num_leaves': 36, 'max_depth': 3, 'feature_fraction': 0.9602213451979265, 'bagging_fraction': 0.7399721557755243, 'min_data_in_leaf': 13}. Best is trial 1 with value: 0.7891788014940725.
[I 2026-08-15 12:58:18,531] Trial 2 finished with value: 0.7629486277269528 and parameters: {'n_estimators': 100, 'learning_rate': 0.07306336069197059, 'num_leaves': 42, 'max_depth': 7, 'feature_fraction': 0.9277907658356659, 'bagging_fraction': 0.8152750096270388, 'min_data_in_leaf': 18}. Best is trial 1 with value: 0.78

In [9]:
lgbm_tuned = LGBMClassifier(**study.best_params, verbose=-1, random_state=RANDOM_STATE)
lgbm_tuned.fit(X_train_s, y_train)

lgbm_tuned_result = evaluate_model('LightGBM (Tuned)', lgbm_tuned, X_test_s, y_test)
all_results.append(lgbm_tuned_result)

print("\nLightGBM (Tuned) Performance:")
for metric, value in lgbm_tuned_result.items():
    if metric != 'Model':
        print(f"  {metric:10s}: {value:.4f}")


LightGBM (Tuned) Performance:
  Accuracy  : 0.7111
  F1        : 0.5000
  ROC-AUC   : 0.7524
  MCC       : 0.3047


## Exercise 2: Tuning Speed vs Quality

Task: Run Optuna with `n_trials=20` and compare to `n_trials=50`

What to check: 
1. How much do best hyperparameters differ?
2. What's the ROC-AUC improvement from 50 vs 20 trials?
3. Is the extra time worth it?

In [10]:
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective_lgbm, n_trials=20, show_progress_bar=True)

print(f"\nBest ROC-AUC (5-fold CV, 20 trials): {study.best_value:.4f}")
print(f"Best Hyperparameters:")
for key, val in sorted(study.best_params.items()):
    print(f"  {key:20s}: {val}")

[I 2026-08-15 12:58:39,476] A new study created in memory with name: no-name-d2613cf1-0eac-4957-9b15-0854aafebc30


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-15 12:58:39,704] Trial 0 finished with value: 0.7614112488496725 and parameters: {'n_estimators': 66, 'learning_rate': 0.015109374061559683, 'num_leaves': 45, 'max_depth': 4, 'feature_fraction': 0.814310541823263, 'bagging_fraction': 0.7123622470798422, 'min_data_in_leaf': 23}. Best is trial 0 with value: 0.7614112488496725.
[I 2026-08-15 12:58:40,262] Trial 1 finished with value: 0.7630826070481243 and parameters: {'n_estimators': 131, 'learning_rate': 0.07894286751748958, 'num_leaves': 22, 'max_depth': 7, 'feature_fraction': 0.7530188688192175, 'bagging_fraction': 0.812056329595104, 'min_data_in_leaf': 18}. Best is trial 1 with value: 0.7630826070481243.
[I 2026-08-15 12:58:40,728] Trial 2 finished with value: 0.7769961565528067 and parameters: {'n_estimators': 107, 'learning_rate': 0.032444173601027956, 'num_leaves': 34, 'max_depth': 5, 'feature_fraction': 0.979563267277902, 'bagging_fraction': 0.7739727802020828, 'min_data_in_leaf': 14}. Best is trial 2 with value: 0.776

## Final Comparison

In [11]:
results_df = pd.DataFrame(all_results).set_index('Model')

print("\nFinal Comparision: RF vs GB vs LightGBM")
print("-"*70)
print(results_df.round(4))
print("-"*70)

print("\nBest Models per Metric:")
for metric in ['Accuracy', 'F1', 'ROC-AUC', 'MCC']:
    best_model = results_df[metric].idxmax()
    best_score = results_df[metric].max()
    print(f"  {metric:10s}: {best_model:30s} ({best_score:.4f})")


Final Comparision: RF vs GB vs LightGBM
----------------------------------------------------------------------
                               Accuracy      F1  ROC-AUC     MCC
Model                                                           
Random Forest (Baseline)         0.7333  0.6000   0.7598  0.4008
Gradient Boosting                0.7222  0.5614   0.7049  0.3584
Gradient Boosting (lr = 0.05)    0.7000  0.5574   0.7275  0.3322
LightGBM (Simple)                0.7000  0.5424   0.6885  0.3194
LightGBM (Tuned)                 0.7111  0.5000   0.7524  0.3047
----------------------------------------------------------------------

Best Models per Metric:
  Accuracy  : Random Forest (Baseline)       (0.7333)
  F1        : Random Forest (Baseline)       (0.6000)
  ROC-AUC   : Random Forest (Baseline)       (0.7598)
  MCC       : Random Forest (Baseline)       (0.4008)


# Summary

## Random Forest vs Gradient Boosting vs LightGBM

| Aspect | Random Forest | Gradient Boosting | LightGBM |
|---|---|---|---|
| Core Idea | Parallel trees | Sequential trees | Leaf-wise boosting |
| Speed | Fast | Medium | Very fast |
| Accuracy | Good baseline | Better | Often best |
| Tuning | Easy | Medium | Medium |
| Best for | Quick baseline | Accuracy-focused | Production |

## Key Insights

1. Sequential learning beats parallel: Boosting corrects errors, bagging averages
2. Hyperparameter tuning matters: Tuned > Untuned > Baseline
3. Use all 4 metrics (no single metric tells the full story)
4. Faster ≠ worse: LightGBM is faster and more accurate

## Chicco & Jurman (2020) Connection

- MCC is most balanced for imbalanced binary classification
- Report multiple metrics: Together they provide comprehensive view
- Ensemble methods consistently beat single models
- Cross-validation is essential for honest performance estimates

# Resources

Gradient Boosting:
- [StatQuest: Gradient Boost](https://www.youtube.com/watch?v=3CC4N_yofrA)
- [StatQuest: XGBoost](https://www.youtube.com/playlist?list=PLblh5JKOoLULnKaL8c4up0HAyABB8GkqR)

LightGBM:
- [LightGBM Official Docs](https://lightgbm.readthedocs.io/)
- [Parameter Tuning Guide](https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html)

Hyperparameter Optimization:
- [Optuna Documentation](https://optuna.readthedocs.io/)
- [Hyperparameter Optimization Strategies](https://www.youtube.com/watch?v=Gbl_Sm7XmVQ)

Evaluation Metrics:
- [Chicco & Jurman (2020): MCC Advantages](https://bmcgenomics.biomedcentral.com/articles/10.1186/s12864-019-6413-7)
- [Imbalanced-Learn Library](https://imbalanced-learn.org/)